# Copia pública del 22 de octubre de 2026

Código de investigación; revisar entradas en `../README.md`. Las salidas guardadas se retiraron. Las etapas 01–06 no se reejecutaron para esta publicación. Para la síntesis numérica utilizada en la memoria, ejecutar `07_Sintesis_resultados_20261022.ipynb`. Las utilidades interactivas son opcionales de revisión y no una dependencia de la síntesis científica.

In [ ]:
from pathlib import Path
import sys
BASE = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'Configuracion.json').is_file()), None)
if BASE is None:
    raise FileNotFoundError('Iniciar Jupyter en notebooks/actual_20261022 o su subcarpeta de notebooks.')
sys.path.insert(0, str(BASE))
import publicacion_config


# 01 · Análisis topográfico
Preparar DEM, hidrología D8, cuencas, redes y perfiles de los **18 outlets**. Mostrar χ para revisar KP y conservar el CSV manual como autoridad.

**Entrega v1 sin ejecutar.** Ejecutar en orden en el kernel geo_stable con TopoAnalysis instalado. Ningún producto nuevo viene calculado. La configuración única está en `Configuracion.json`; θ=0,40, A₀=10⁶ m², ventana base=2000 m. La sensibilidad incluye 250, 500, 750, 1000 y 2000 m.

Las elevaciones provienen del DEM original; el DEM acondicionado solo define el drenaje. Cambiar `run_id` a `v2` antes de recalcular una versión científica diferente. Las figuras reciben versiones propias, sin fecha en el nombre.

In [ ]:
from pathlib import Path
import sys
BASE = publicacion_config.base()
if not (BASE/'Configuracion.json').is_file():
    BASE = Path(str(BASE / 'inputs_external' / 'Jupyter_Memoria'))
if not (BASE/'Configuracion.json').is_file():
    raise FileNotFoundError('Abrir Jupyter desde la carpeta Jupyter_Memoria.')
CARPETA_NOTEBOOKS = BASE
CARPETA_CODIGO = BASE/'Archivos_Memoria/Codigo'
sys.path.insert(0,str(CARPETA_CODIGO))
import importlib
import memoria_comun as pa
pa=importlib.reload(pa)
CFG,RUN_ID=pa.CFG,pa.RUN_ID
sys.path.insert(0,str(Path(CFG['topoanalysis_codigo']).parent))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
plt.rcParams.update({'pdf.fonttype':42,'font.family':'DejaVu Sans','font.size':10})
print('Corrida compartida:',RUN_ID,'| θ:',CFG['theta'],'| ventana:',CFG['ventana_m'],'m')

pa.NOTEBOOK_PATH=BASE/'Notebooks_principales_20261019/01_Analisis_Topografico.ipynb'

## Configuración y catálogo único

In [ ]:
from TopoAnalysis import dem as d
from pathlib import Path
from datetime import datetime
import json
import sys
import importlib.metadata as metadata

import numpy as np
import pandas as pd
import rasterio
from rasterio.features import shapes
from rasterio.windows import Window
import geopandas as gpd
from shapely.geometry import shape

BASE_DIR = Path(CFG['base_dir'])
DEM_PATH = Path(CFG['dem'])
FECHA = RUN_ID
DATA_DIR = pa.ROOT / 'topografia'
MASK_DIR = DATA_DIR / 'masks'
POLY_DIR = DATA_DIR / 'basin_polygons'
CRS_OUTLETS = rasterio.crs.CRS.from_string(CFG['crs'])
GUARDAR_LOGAREA = False
REUTILIZAR_HIDROLOGIA = True

In [ ]:
catalogo = pd.read_csv(CFG['outlets_csv'])
catalogo['rio_id'] = catalogo['rio_id'].map(pa.normalize_id)
if len(catalogo) != 18 or set(catalogo['rio_id']) != set(pa.EXPECTED):
    raise ValueError('Se requieren los 18 IDs, sin duplicados.')
if catalogo['nombre'].duplicated().any() or catalogo[['x', 'y']].duplicated().any():
    raise ValueError('Catálogo de outlets duplicado.')
if not np.isfinite(catalogo[['x', 'y']].to_numpy(float)).all():
    raise ValueError('Coordenadas inválidas.')
if not catalogo['crs'].eq(CFG['crs']).all(): raise ValueError('CRS del catálogo incompatible.')
samples = catalogo[['nombre', 'x', 'y']].values.tolist()
display(catalogo)

## Validación de entradas antes del cálculo

In [ ]:
if not DEM_PATH.is_file():
    raise FileNotFoundError(f'No existe el DEM: {DEM_PATH}')

controles = []
with rasterio.open(DEM_PATH) as src:
    if src.count != 1:
        raise ValueError('Se esperaba un DEM de una sola banda.')
    if src.crs is None or not src.crs.is_projected:
        raise ValueError('El DEM debe tener un CRS proyectado definido.')
    if not np.isclose(src.crs.linear_units_factor[1], 1.0):
        raise ValueError('Las unidades horizontales deben ser metros.')
    if src.crs != CRS_OUTLETS:
        raise ValueError(f'CRS incompatibles: DEM={src.crs}; outlets={CRS_OUTLETS}.')

    transform_dem = src.transform
    if (transform_dem.b != 0 or transform_dem.d != 0
            or transform_dem.a <= 0 or transform_dem.e >= 0
            or not np.isclose(transform_dem.a, -transform_dem.e, rtol=1e-9, atol=1e-9)):
        raise ValueError('Se requiere una grilla al norte, sin rotación y con píxeles cuadrados.')

    crs_dem = src.crs
    shape_dem = src.shape
    resolucion_m = float(transform_dem.a)
    area_celda_m2 = abs(transform_dem.a * transform_dem.e - transform_dem.b * transform_dem.d)
    n_total = src.width * src.height
    n_validas = 0
    for _, ventana in src.block_windows(1):
        bloque = src.read(1, window=ventana, masked=True)
        validas = ~np.ma.getmaskarray(bloque) & np.isfinite(bloque.data)
        n_validas += int(np.count_nonzero(validas))
    del bloque, validas
    if n_validas == 0:
        raise ValueError('El DEM no contiene elevaciones válidas.')

    xllcenter = transform_dem.c + resolucion_m / 2
    yllcenter = transform_dem.f - resolucion_m * (src.height - 0.5)
    for rio in catalogo.itertuples(index=False):
        columna = int(round((rio.x - xllcenter) / resolucion_m))
        fila = int((src.height - 1) - round((rio.y - yllcenter) / resolucion_m))
        detalle = ''
        if not (0 <= fila < src.height and 0 <= columna < src.width):
            detalle = 'Outlet fuera de la grilla'
        else:
            valor = src.read(1, window=Window(columna, fila, 1, 1), masked=True)
            if valor.count() != 1 or not np.isfinite(valor.compressed()).all():
                detalle = 'Outlet sin elevación válida'
        controles.append({'rio_id': rio.rio_id, 'nombre': rio.nombre,
                          'fila': fila, 'columna': columna,
                          'estado': 'error' if detalle else 'ok', 'detalle': detalle})

for carpeta in (DATA_DIR, MASK_DIR, POLY_DIR):
    carpeta.mkdir(parents=True, exist_ok=True)
catalogo.to_csv(DATA_DIR / f'catalogo_outlets_{FECHA}.csv', index=False, encoding='utf-8-sig')
qc_outlets = pd.DataFrame(controles)
qc_outlets.to_csv(DATA_DIR / f'qc_outlets_{FECHA}.csv', index=False, encoding='utf-8-sig')
if not qc_outlets['estado'].eq('ok').all():
    display(qc_outlets)
    raise ValueError('Hay outlets inválidos. Consultar qc_outlets antes de continuar.')

print('CRS:', crs_dem)
print(f'Resolución: {resolucion_m:g} m')
print(f'Celdas válidas: {n_validas:,} / {n_total:,}')
print(f'Sin datos: {100 * (1 - n_validas / n_total):.2f}%')
print('Los 18 outlets tienen cobertura válida.')

## Identidad de la hidrología y recuperación opcional
La huella del DEM se calcula leyendo el archivo, sin modificarlo. La caché solo reutiliza productos registrados y sin cambios; una discrepancia obliga a recalcular.

In [ ]:
from importlib import metadata
codigo_topo = Path(d.__file__).parent
implementacion = {str(p.relative_to(codigo_topo)): pa.file_hash(p)
                  for p in sorted(codigo_topo.rglob('*'))
                  if p.is_file() and p.suffix.lower() in {'.py', '.pyd', '.so'}}
versiones_cache = {}
for paquete in ['numpy', 'scipy', 'GDAL', 'TopoAnalysis']:
    try: versiones_cache[paquete] = metadata.version(paquete)
    except metadata.PackageNotFoundError: versiones_cache[paquete] = 'sin_metadatos'
cache_key = pa.signature({'dem': pa.file_hash(DEM_PATH), 'codigo': implementacion,
    'versiones': versiones_cache, 'python': sys.version, 'metodo': 'FilledElevation_defaults_D8_Area',
    'aggradation_slope': getattr(d.FilledElevation, 'aggradation_slope', None)})
recuperar = False
if REUTILIZAR_HIDROLOGIA and pa.output('topografia', 'manifest', 'json').exists():
    try:
        previo = pa.read_stage('topografia')
        recuperar = previo.get('cache_key') == cache_key
    except (ValueError, OSError, KeyError) as exc:
        print('Caché no reutilizable:', exc)
pa.begin('topografia')
print('Recuperar hidrología:', recuperar)

## Cargar o calcular las grillas

In [ ]:
if recuperar:
    filled = d.FilledElevation.load(previo['files']['filled']['path'])
    flow_direction = d.FlowDirectionD8.load(previo['files']['flow_direction']['path'])
    area = d.Area.load(previo['files']['area']['path'])
else:
    elevation = d.Elevation.load(str(DEM_PATH))
    if elevation._griddata.shape != shape_dem:
        raise ValueError('Las dimensiones cargadas no coinciden con el DEM.')

    indices_topo = elevation._xy_to_rowscols(catalogo[['x', 'y']].itertuples(index=False, name=None))
    indices_qc = list(qc_outlets[['fila', 'columna']].itertuples(index=False, name=None))
    if list(indices_topo) != indices_qc:
        raise ValueError('La conversión de coordenadas difiere del control previo; revisar TopoAnalysis.')
    print('DEM original cargado.')
    filled = d.FilledElevation(elevation=elevation)
    print('Relleno terminado.')
    flow_direction = d.FlowDirectionD8(flooded_dem=filled)
    print('Dirección D8 terminada.')
    area = d.Area(flow_direction=flow_direction)
    print('Área aportante terminada.')
    log_area = None
    if GUARDAR_LOGAREA:
        log_area = d.LogArea(area=area)
        print('Logaritmo del área terminado.')
log_area = d.LogArea(area=area) if GUARDAR_LOGAREA else None

## Guardar y verificar grillas

In [ ]:
productos = {
    'filled': (filled, DATA_DIR / f'dem_filled_{resolucion_m:g}m_{FECHA}.tif'),
    'flow_direction': (flow_direction, DATA_DIR / f'flow_direction_D8_{resolucion_m:g}m_{FECHA}.tif'),
    'area': (area, DATA_DIR / f'area_aportante_{resolucion_m:g}m_{FECHA}.tif'),
}
if GUARDAR_LOGAREA:
    productos['log_area'] = (log_area, DATA_DIR / f'logarea_{resolucion_m:g}m_{FECHA}.tif')

rutas_productos = {}
for nombre, (grilla, ruta) in productos.items():
    grilla.save(str(ruta))
    with rasterio.open(ruta) as src:
        if (src.crs != crs_dem or src.shape != shape_dem
                or not src.transform.almost_equals(transform_dem)):
            raise ValueError(f'Georreferenciación incompatible: {ruta.name}')
    rutas_productos[nombre] = str(ruta)
    print('Guardado:', ruta.name)

versiones = {}
for paquete in ('TopoAnalysis', 'numpy', 'rasterio', 'geopandas', 'pandas', 'shapely'):
    try:
        versiones[paquete] = metadata.version(paquete)
    except metadata.PackageNotFoundError:
        versiones[paquete] = 'sin metadatos de distribución'

manifiesto = {
    'fecha_ejecucion': FECHA,
    'dem_original': str(DEM_PATH),
    'dem_size_bytes': DEM_PATH.stat().st_size,
    'dem_mtime_ns': DEM_PATH.stat().st_mtime_ns,
    'crs': crs_dem.to_string(),
    'transformacion': list(transform_dem)[:6],
    'dimensiones': list(shape_dem),
    'resolucion_m': resolucion_m,
    'area_celda_m2': area_celda_m2,
    'celdas_validas_dem': n_validas,
    'metodo': 'FilledElevation -> FlowDirectionD8 -> Area',
    'parametros': 'Valores por defecto de la implementación instalada',
    'topoanalysis_fuente': str(Path(d.__file__).resolve()),
    'python': sys.version,
    'ejecutable': sys.executable,
    'versiones': versiones,
    'productos': rutas_productos,
    'catalogo_outlets': str(DATA_DIR / f'catalogo_outlets_{FECHA}.csv'),
    'estado_cuencas': 'pendiente',
}
ruta_manifiesto = DATA_DIR / f'manifiesto_topografico_{FECHA}.json'
ruta_manifiesto.write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding='utf-8')

## Procesar todas las cuencas y registrar sus estados

In [ ]:
estados = []
ruta_resumen = DATA_DIR / f'resumen_cuencas_{FECHA}.csv'
manifiesto['estado_cuencas'] = 'en_proceso'
ruta_manifiesto.write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding='utf-8')

for rio in catalogo.itertuples(index=False):
    estado = {'rio_id': rio.rio_id, 'nombre': rio.nombre, 'x': rio.x, 'y': rio.y,
              'estado': 'error', 'area_km2': np.nan, 'n_celdas': 0,
              'mascara': '', 'poligono': '', 'detalle': ''}
    basin_mask = image = positivas = gdf = geometrias = None
    try:
        fila, columna = flow_direction._xy_to_rowscols([(rio.x, rio.y)])[0]
        if fila is None or columna is None:
            raise ValueError('Outlet fuera de la grilla.')
        prefijo = f'rio_{int(rio.rio_id):02d}'
        ruta_mask = MASK_DIR / f'{prefijo}_mask_{FECHA}.tif'
        ruta_poly = POLY_DIR / f'{prefijo}_cuenca_{FECHA}.gpkg'

        basin_mask = d.Mask(flow_direction=flow_direction, outlets=((rio.x, rio.y),))
        basin_mask.save(str(ruta_mask))
        estado['mascara'] = str(ruta_mask)
        basin_mask = None

        with rasterio.open(ruta_mask) as src:
            if src.crs != crs_dem:
                raise ValueError('El CRS de la máscara no coincide con el DEM.')
            if src.shape != shape_dem or not src.transform.almost_equals(transform_dem):
                raise ValueError('La máscara no está alineada con el DEM.')
            image = src.read(1, masked=True)
            positivas = (~np.ma.getmaskarray(image)
                         & np.isfinite(image.data) & (image.data > 0))
            n_celdas = int(np.count_nonzero(positivas))
            if n_celdas == 0:
                raise ValueError('La cuenca está vacía.')
            if not positivas[fila, columna]:
                raise ValueError('La máscara no incluye el outlet.')
            area_m2 = n_celdas * area_celda_m2
            geometrias = [shape(geom) for geom, _ in shapes(
                positivas.astype('uint8'), mask=positivas, transform=src.transform)]

        gdf = gpd.GeoDataFrame({'rio_id': [rio.rio_id] * len(geometrias)},
                               geometry=geometrias, crs=crs_dem)
        if gdf.empty or not gdf.geometry.is_valid.all():
            raise ValueError('Se generaron geometrías vacías o inválidas.')
        if not np.isclose(float(gdf.geometry.area.sum()), area_m2, rtol=1e-8, atol=0.01):
            raise ValueError('El área vectorial no coincide con la máscara.')
        gdf.to_file(ruta_poly, layer='cuenca', driver='GPKG', mode='w')
        estado.update(estado='ok', area_km2=area_m2 / 1e6,
                      n_celdas=n_celdas, poligono=str(ruta_poly))
        print(f'{int(rio.rio_id):02d} | {rio.nombre}: {area_m2 / 1e6:.2f} km²')
    except Exception as exc:
        estado['detalle'] = f'{type(exc).__name__}: {exc}'
        print(f'{int(rio.rio_id):02d} | {rio.nombre}: ERROR — {exc}')
    finally:
        estados.append(estado)
        pd.DataFrame(estados).to_csv(ruta_resumen, index=False, encoding='utf-8-sig')
        basin_mask = image = positivas = gdf = geometrias = None

resumen = pd.DataFrame(estados)
fallidos = resumen.loc[resumen['estado'] != 'ok']
manifiesto.update(
    estado_cuencas='completo' if fallidos.empty else 'incompleto',
    resumen_cuencas=str(ruta_resumen),
    cuencas_correctas=int(resumen['estado'].eq('ok').sum()),
    cuencas_esperadas=len(catalogo),
)
ruta_manifiesto.write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding='utf-8')
display(resumen[['rio_id', 'nombre', 'estado', 'area_km2', 'detalle']])
print('Resumen:', ruta_resumen)
if not fallidos.empty:
    raise RuntimeError('Hay cuencas pendientes. Consultar la columna detalle del resumen.')

In [ ]:
archivos = {**rutas_productos, 'outlets': DATA_DIR / f'catalogo_outlets_{FECHA}.csv',
            'qc_outlets': DATA_DIR / f'qc_outlets_{FECHA}.csv', 'resumen': ruta_resumen,
            'procedencia': ruta_manifiesto}
for row in resumen.itertuples(index=False):
    archivos[f'mascara_{row.rio_id}'] = row.mascara
    archivos[f'poligono_{row.rio_id}'] = row.poligono
pa.finish('topografia', archivos, extra={'cache_key': cache_key, 'crs': CFG['crs'],
    'dem_original': str(DEM_PATH), 'dem_sha256': pa.file_hash(DEM_PATH)})
print('Topografía completa para esta corrida:', RUN_ID)

## Extraer una vez todos los perfiles
El cauce principal sigue el tributario de mayor área. χ se integra por trapecios sobre la distancia acumulada real entre centros de celda. El ksn local usa regresión S–distancia y se invalida donde S≤0 o hay huecos. La selección para dibujar no reduce el catálogo calculado.

In [ ]:
from TopoAnalysis import dem as d
import rasterio
from rasterio.enums import Resampling
from scipy.stats import linregress
topo = pa.read_stage('topografia')
deps = pa.dependency('topografia')
DEM_PATH = Path(CFG['dem'])
if pa.file_hash(DEM_PATH) != topo['dem_sha256']: raise ValueError('El DEM cambió desde topografía.')
deps[str(DEM_PATH)] = topo['dem_sha256']
AREA_PATH = Path(topo['files']['area']['path'])
FD_PATH = Path(topo['files']['flow_direction']['path'])
catalogo = pd.read_csv(topo['files']['outlets']['path'], dtype={'rio_id': str})
river_ids = dict(zip(catalogo.rio_id, catalogo.nombre))
sample_outlets = {r.nombre: (r.x, r.y) for r in catalogo.itertuples(index=False)}
A0, AREA_THRESHOLD, THETA_REF = CFG['A0_m2'], CFG['area_minima_m2'], CFG['theta']
colors = {name: plt.get_cmap('tab20')(i) for i, name in enumerate(sample_outlets)}
pa.begin('perfiles')

In [ ]:
# Comprobar que existan las entradas
for path in (AREA_PATH, FD_PATH, DEM_PATH):
    if not path.exists():
        raise FileNotFoundError(f"No se encontró: {path}")


# ── Cargar las grillas con TopoAnalysis ───────────────────────────────────
area = d.Area.load(str(AREA_PATH))
fd = d.FlowDirectionD8.load(str(FD_PATH))
elevation = d.Elevation.load(str(DEM_PATH))


# ── Acceder a las matrices ───────────────────────────────────────────────
area_grid = area._griddata
fd_grid = fd._griddata
elev_grid = elevation._griddata


# ── Comprobar dimensiones ────────────────────────────────────────────────
if not (area_grid.shape == fd_grid.shape == elev_grid.shape):
    raise ValueError(
        "Las grillas tienen dimensiones diferentes:\n"
        f"Área: {area_grid.shape}\n"
        f"Dirección de flujo: {fd_grid.shape}\n"
        f"Elevación: {elev_grid.shape}"
    )

print("Grillas cargadas correctamente.")
print(f"Dimensiones: {elev_grid.shape[0]} filas × "
      f"{elev_grid.shape[1]} columnas")
print(f"DEM de elevación: {DEM_PATH.name}")
print("Siguiente paso: comprobar la alineación espacial y los outlets.")

In [ ]:
import rasterio

# ── Leer la información espacial ─────────────────────────────────────────
grid_info = {}

for name, path in {
    "Elevación": DEM_PATH,
    "Área": AREA_PATH,
    "Dirección": FD_PATH,
}.items():
    with rasterio.open(path) as src:
        grid_info[name] = {
            "crs": src.crs,
            "transform": src.transform,
            "shape": src.shape,
            "res": src.res,
        }

reference = grid_info["Elevación"]

if reference["crs"] is None:
    raise ValueError("El DEM no tiene un sistema de coordenadas definido.")

if not reference["crs"].is_projected:
    raise ValueError("El DEM debe estar en coordenadas proyectadas.")

for name, info in grid_info.items():
    if info["crs"] != reference["crs"]:
        raise ValueError(f"{name}: sistema de coordenadas diferente.")

    if info["shape"] != reference["shape"]:
        raise ValueError(f"{name}: dimensiones diferentes.")

    if not info["transform"].almost_equals(reference["transform"]):
        raise ValueError(f"{name}: alineación o resolución diferente.")

print("Las tres grillas coinciden espacialmente.")
print(f"Sistema de coordenadas: {reference['crs']}")
print(f"Resolución: {reference['res']}")
print(f"Unidad: {reference['crs'].linear_units}")

## Extraer una sola vez y calcular todos los IDs
Una extracción fallida queda en el resumen. Una exportación incompleta no se autoriza como entrada de las etapas siguientes.

In [ ]:
# ── Función para extraer un perfil longitudinal ──────────────────────────
def extraer_perfil(outlet, area_minima=AREA_THRESHOLD):
    row, col = fd._xy_to_rowscols((outlet,))[0]

    if row is None or col is None:
        raise ValueError("El outlet está fuera de la grilla.")

    nrows, ncols = area_grid.shape
    recorrido = []
    visitadas = set()

    while True:
        if (row, col) in visitadas:
            raise ValueError("Se detectó un ciclo en la dirección de flujo.")

        visitadas.add((row, col))
        drainage_area = float(area_grid[row, col])

        if not np.isfinite(drainage_area) or drainage_area < area_minima:
            if not recorrido:
                raise ValueError(
                    "El outlet no tiene área válida o está bajo el umbral."
                )
            break

        recorrido.append((row, col))

        # Vecinos que realmente drenan hacia la celda actual
        candidatos = []

        for r, c in fd.get_upstream_cell_indexes(row, col):
            r, c = int(r), int(c)

            if 0 <= r < nrows and 0 <= c < ncols:
                a = float(area_grid[r, c])

                if np.isfinite(a) and a >= area_minima:
                    candidatos.append((r, c))

        if not candidatos:
            break

        # Criterio de cauce principal: mayor área aportante
        row, col = max(
            candidatos,
            key=lambda rc: float(area_grid[rc[0], rc[1]])
        )

    if len(recorrido) < 2:
        raise ValueError(
            "El perfil tiene menos de dos puntos. Revisa el outlet y el umbral."
        )

    rows, cols = np.array(recorrido, dtype=int).T
    xy = np.asarray(fd._rowscols_to_xy(recorrido), dtype=float)
    x, y = xy[:, 0], xy[:, 1]

    # Distancia real entre centros de celdas, incluidas las diagonales
    pasos = np.hypot(np.diff(x), np.diff(y))
    distancia = np.concatenate(([0.0], np.cumsum(pasos)))

    perfil = pd.DataFrame({
        "fila": rows,
        "columna": cols,
        "x_utm": x,
        "y_utm": y,
        "distancia_m": distancia,
        "elevacion_m": elev_grid[rows, cols],
        "area_m2": area_grid[rows, cols],
    })

    # Conservar las posiciones sin elevación para no unir huecos en el gráfico
    perfil["elevacion_valida"] = np.isfinite(perfil["elevacion_m"])
    perfil.loc[~perfil["elevacion_valida"], "elevacion_m"] = np.nan

    # Diferencia entre la coordenada indicada y el centro de su celda
    perfil.attrs["ajuste_a_celda_m"] = float(
        np.hypot(x[0] - outlet[0], y[0] - outlet[1])
    )

    return perfil

In [ ]:
perfiles_ksn, sensibilidad_ksn, estados = {}, {}, []
archivos, rios = {}, {}
for rid in pa.EXPECTED:
    registro = {'ID': rid, 'rio': river_ids[rid], 'estado': 'error', 'detalle': ''}
    try:
        p = extraer_perfil(sample_outlets[river_ids[rid]], area_minima=AREA_THRESHOLD)
        p['chi_m'] = pa.chi_1d(p)
        # Diagnóstico de discretización, sin afirmar que reproduce d.Chi.
        s, a = p['distancia_m'].to_numpy(float), p['area_m2'].to_numpy(float)
        rect = np.r_[0., np.cumsum(np.diff(s)*(A0/a[:-1])**THETA_REF)]
        p.attrs.update(theta_ref=THETA_REF, A0_m2=A0, area_minima_m2=AREA_THRESHOLD,
                       metodo_chi=CFG['metodo_chi'], run_id=RUN_ID)
        base = pa.slope_ksn(p, CFG['ventana_m'], CFG['min_puntos'])
        perfiles_ksn[rid] = base
        for window in sorted(set([CFG['ventana_m'], *CFG['ventanas_sensibilidad_m']])):
            result = base if window == CFG['ventana_m'] else pa.slope_ksn(p, window, CFG['min_puntos'])
            sensibilidad_ksn[(rid, window)] = result
            archivos[f'sens_{rid}_{window:g}'] = pa.save_table('perfiles', f'sens_rio_{int(rid):02d}_{window:g}m', result)
        archivos[f'rio_{rid}'] = pa.save_table('perfiles', f'rio_{int(rid):02d}', base)
        rios[rid] = {'nombre': river_ids[rid], 'archivo': archivos[f'rio_{rid}'].name,
                     'parametros': dict(base.attrs), 'perfil_hash': pa.profile_hash(base)}
        registro.update(estado='ok', puntos=len(p), sin_elevacion=int(p.elevacion_m.isna().sum()),
            ksn_validos=int(base.ksn_valido.sum()), pendientes_no_positivas=int(base.pendiente_no_positiva.sum()),
            ventanas_parciales=int(base.ventana_parcial.sum()),
            diferencia_trapecio_rectangulo_aval_m=float(np.max(np.abs(p.chi_m.to_numpy()-rect))))
    except (ValueError, IndexError, RuntimeError) as exc:
        registro['detalle'] = str(exc)
    estados.append(registro)
    archivos['qc'] = pa.save_table('perfiles', 'qc_perfiles', pd.DataFrame(estados))
resumen_qc = pd.DataFrame(estados)
display(resumen_qc)
if set(perfiles_ksn) != set(pa.EXPECTED):
    raise RuntimeError('Faltan perfiles: revisar qc_perfiles antes de continuar.')

## Exportar cauces principales y registrar la corrida
La red de todos los tributarios es un producto distinto. Aquí se guarda exclusivamente el cauce principal de cada outlet, sin volver a extraerlo.

In [ ]:
import geopandas as gpd
from shapely.geometry import LineString
cauces = gpd.GeoDataFrame([{'ID': rid, 'rio': river_ids[rid], 'perfil_hash': pa.profile_hash(p),
    'geometry': LineString(p[['x_utm','y_utm']].to_numpy(float))} for rid,p in perfiles_ksn.items()], crs=CFG['crs'])
ruta_cauces = pa.output('perfiles', 'cauces_principales', 'gpkg')
cauces.to_file(ruta_cauces, layer='cauces', driver='GPKG', mode='w')
archivos['cauces_principales'] = ruta_cauces
pa.finish('perfiles', archivos, deps, extra={'rios': rios, 'unidad_distancia': 'm', 'unidad_area': 'm2',
    'metodo_chi': CFG['metodo_chi'], 'criterio_cauce': 'Mayor área aportante en cada confluencia'})

## Red completa y cauces principales
Exportación de todas las conexiones D8 con área ≥ umbral, reutilizando los perfiles extraídos. El control verifica que cada conexión se exporte una sola vez.

In [ ]:
# ── Exportar red completa y cauces principales de los outlets ──────────
from pathlib import Path
from datetime import datetime
from collections import Counter
import numpy as np
import rasterio
from osgeo import ogr, osr

# CONFIGURACIÓN
RUTA_REDES = BASE/'Productos/01_Topografia/Redes'

AREA_MIN_RED = float(AREA_THRESHOLD)  # m²; actualmente 1 km²
RIOS_EXPORTAR = list(river_ids)      # Los 18 outlets
# RIOS_EXPORTAR = ["6", "10"]        # Alternativa: solo estos ríos

# ── Validaciones ──────────────────────────────────────────────────────
if not np.isfinite(AREA_MIN_RED) or AREA_MIN_RED <= 0:
    raise ValueError("AREA_MIN_RED debe ser positiva y finita.")

ids_red = list(dict.fromkeys(str(i) for i in RIOS_EXPORTAR))
if not ids_red or any(i not in river_ids for i in ids_red):
    raise ValueError("Selecciona IDs válidos en RIOS_EXPORTAR.")

with rasterio.open(AREA_PATH) as src:
    crs_red = src.crs
    mascara_red = (
        (src.read_masks(1) > 0)
        & np.isfinite(area_grid)
        & (area_grid >= AREA_MIN_RED)
    )

if crs_red is None or crs_red.to_epsg() != 32718:
    raise ValueError("Se esperaba WGS 84 / UTM 18S, EPSG:32718.")

nfilas, ncolumnas = area_grid.shape

# Extraer en variables nuevas: no reemplazar perfiles, χ ni ksn.
principales_red = perfiles_ksn

# ── Construir las conexiones de la red ───────────────────────────────
# Se usa la función de TopoAnalysis del propio notebook.
# No se presupone una codificación D8 externa.
celdas_red = np.flatnonzero(mascara_red.ravel())
receptor_red = {}  # celda aguas arriba -> celda aguas abajo

print(
    f"Construyendo red: {len(celdas_red):,} celdas "
    f"con área ≥ {AREA_MIN_RED / 1e6:g} km²..."
)

for posicion, celda in enumerate(celdas_red, start=1):
    celda = int(celda)
    fila, columna = divmod(celda, ncolumnas)

    for r, c in fd.get_upstream_cell_indexes(fila, columna):
        r, c = int(r), int(c)

        if (
            0 <= r < nfilas
            and 0 <= c < ncolumnas
            and mascara_red[r, c]
        ):
            aguas_arriba = r * ncolumnas + c

            if (
                aguas_arriba in receptor_red
                and receptor_red[aguas_arriba] != celda
            ):
                raise ValueError("Una celda tiene dos receptores D8.")

            receptor_red[aguas_arriba] = celda

    if posicion % 100000 == 0:
        print(f"  Revisadas {posicion:,} celdas.")

if not receptor_red:
    raise ValueError("No se encontraron conexiones con este umbral.")

# Una línea por tramo entre cabeceras, confluencias y terminaciones.
numero_afluentes = Counter(receptor_red.values())
inicios_red = [
    celda for celda in receptor_red
    if numero_afluentes[celda] != 1
]

# ── Preparar salida y sistema de coordenadas ─────────────────────────
carpeta_red = pa.nueva_carpeta(RUTA_REDES)
carpeta_red.mkdir(parents=True, exist_ok=True)

srs_red = osr.SpatialReference()
srs_red.ImportFromWkt(crs_red.to_wkt())
srs_red.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)

ogr.UseExceptions()
driver_red = ogr.GetDriverByName("ESRI Shapefile")

if driver_red is None:
    raise RuntimeError("GDAL no tiene disponible el controlador Shapefile.")


def crear_shp_red(ruta, campos):
    ds = driver_red.CreateDataSource(str(ruta))
    capa = ds.CreateLayer(
        ruta.stem,
        srs=srs_red,
        geom_type=ogr.wkbLineString,
        options=["ENCODING=UTF-8"],
    )

    for nombre, tipo in campos:
        campo = ogr.FieldDefn(nombre, tipo)
        if tipo == ogr.OFTString:
            campo.SetWidth(100)
        capa.CreateField(campo)

    return ds, capa


def agregar_linea_red(capa, xy, atributos):
    xy = np.asarray(xy, dtype=float)

    if len(xy) < 2 or not np.isfinite(xy).all():
        raise ValueError("Geometría con menos de dos puntos o coordenadas inválidas.")

    geometria = ogr.Geometry(ogr.wkbLineString)
    for x, y in xy:
        geometria.AddPoint_2D(float(x), float(y))

    registro = ogr.Feature(capa.GetLayerDefn())
    registro.SetGeometry(geometria)

    for nombre, valor in atributos.items():
        registro.SetField(nombre, valor)

    capa.CreateFeature(registro)
    registro = None


# ── 1. Red completa del DEM ──────────────────────────────────────────
ruta_completa = carpeta_red / "red_completa.shp"

ds_red, capa_red = crear_shp_red(
    ruta_completa,
    [
        ("tramo_id", ogr.OFTInteger),
        ("long_m", ogr.OFTReal),
        ("amin_km2", ogr.OFTReal),
    ],
)

enlaces_exportados = set()
ntramos_red = 0

try:
    for inicio in inicios_red:
        recorrido = [inicio]
        actual = inicio

        while actual in receptor_red:
            if actual in enlaces_exportados:
                raise ValueError("Conexión repetida o ciclo en la red.")

            enlaces_exportados.add(actual)
            siguiente = receptor_red[actual]
            recorrido.append(siguiente)

            if numero_afluentes[siguiente] != 1:
                break

            actual = siguiente

        filas_columnas = [
            divmod(celda, ncolumnas) for celda in recorrido
        ]
        xy = np.asarray(
            fd._rowscols_to_xy(filas_columnas), dtype=float
        )
        longitud = float(np.hypot(
            np.diff(xy[:, 0]), np.diff(xy[:, 1])
        ).sum())

        ntramos_red += 1
        agregar_linea_red(
            capa_red,
            xy,
            {
                "tramo_id": ntramos_red,
                "long_m": longitud,
                "amin_km2": AREA_MIN_RED / 1e6,
            },
        )

    if len(enlaces_exportados) != len(receptor_red):
        raise ValueError(
            "Quedaron conexiones sin exportar. Posible ciclo D8; "
            "no utilices esta exportación hasta revisarlo."
        )
finally:
    capa_red = None
    ds_red = None

# ── 2. Cauces principales asociados a tus outlets ────────────────────
ruta_outlets = carpeta_red / "rios_outlets.shp"

ds_red, capa_red = crear_shp_red(
    ruta_outlets,
    [
        ("rio_id", ogr.OFTString),
        ("rio", ogr.OFTString),
        ("long_m", ogr.OFTReal),
        ("amin_km2", ogr.OFTReal),
        ("ajuste_m", ogr.OFTReal),
    ],
)

try:
    for rid, perfil_red in principales_red.items():
        # El perfil original comienza en el outlet.
        # Invertimos solo la geometría exportada para orientarla aguas abajo.
        xy = perfil_red[["x_utm", "y_utm"]].to_numpy(dtype=float)[::-1]

        agregar_linea_red(
            capa_red,
            xy,
            {
                "rio_id": rid,
                "rio": river_ids[rid],
                "long_m": float(perfil_red["distancia_m"].iloc[-1]),
                "amin_km2": AREA_MIN_RED / 1e6,
                "ajuste_m": float(
                    perfil_red.attrs["ajuste_a_celda_m"]
                ),
            },
        )
finally:
    capa_red = None
    ds_red = None

# ── Verificar que ambos archivos puedan abrirse ──────────────────────
for ruta, cantidad in (
    (ruta_completa, ntramos_red),
    (ruta_outlets, len(principales_red)),
):
    ds_revision = ogr.Open(str(ruta))
    if ds_revision is None:
        raise RuntimeError(f"No se pudo abrir el resultado: {ruta}")

    capa_revision = ds_revision.GetLayer()
    if capa_revision.GetFeatureCount() != cantidad:
        raise RuntimeError(f"Número de geometrías incorrecto: {ruta}")

    capa_revision = None
    ds_revision = None

print("\nExportación terminada.")
print(f"Red completa: {ntramos_red:,} tramos.")
print(f"Cauces principales: {len(principales_red)} ríos.")
print(f"Umbral: {AREA_MIN_RED / 1e6:g} km².")
print("CRS: EPSG:32718.")
print(f"\nCarpeta:\n{carpeta_red}")

## Revisión de χ y selecciones manuales
Las estrellas negras son KP del CSV verificados por posición y elevación; un río sin selección conserva su estado. Cambiar `RIOS_VER` afecta únicamente los paneles.

No se aceptan candidatos automáticos como KP definitivos. Las selecciones originales se preservan; el editor opcional del bloque siguiente guarda una copia nueva y actualiza la configuración compartida.

In [ ]:
perfiles,nombres,manifest=pa.load_profiles()
manuales,rechazados=pa.load_manuals(perfiles)
RIOS_VER=list(perfiles)
for rid in RIOS_VER:
    p=perfiles[rid]
    fig,ax=plt.subplots(figsize=(10,4),layout='constrained')
    ax.plot(p.chi_m/1000,p.elevacion_m,lw=1.2)
    for _,kp in manuales.loc[manuales.ID.eq(rid)].iterrows():
        ax.scatter(np.interp(kp.distancia_m,p.distancia_m,p.chi_m)/1000,kp.elevacion_m,
                   marker='*',color='black',s=80,zorder=4)
    ax.set(title=f'{int(rid):02d} · {nombres[rid]}',xlabel='χ (km)',ylabel='Elevación (m)')
    pa.guardar_figura(fig,'Chi_revision',rid,'01_Topografia')
    plt.show();plt.close(fig)
display(manuales)
display(rechazados)

## Editor opcional de KP
Activar únicamente para cambiar la selección. Se abre un perfil χ por río, se elige un punto y se ajusta al nodo más cercano del gráfico. Cerrar sin marcar conserva la selección. Guarda un CSV nuevo; los originales permanecen intactos. Después de editar, recalcular los notebooks 02–04.

In [ ]:
EDITAR_KP=False
RIOS_EDITAR=['6']
if EDITAR_KP:
    import json
    from datetime import datetime,timezone
    get_ipython().run_line_magic('matplotlib','qt')
    tabla=pd.read_csv(CFG['kp_manual_csv'],sep=';',decimal=',',dtype={'ID':str}).dropna(how='all')
    tabla['ID']=[pa.normalize_id(r.ID if pd.notna(r.ID) else r['id']) for _,r in tabla.iterrows()]
    for rid in map(str,RIOS_EDITAR):
        p=perfiles[rid]
        fig,ax=plt.subplots(figsize=(11,5))
        ax.plot(p.chi_m,p.elevacion_m);ax.set(title=nombres[rid],xlabel='χ (m)',ylabel='Elevación (m)')
        seleccion=plt.ginput(1,timeout=-1);plt.close(fig)
        if not seleccion:continue
        x,z=seleccion[0]
        dx=max(float(np.ptp(p.chi_m)),1.0);dz=max(float(np.nanmax(p.elevacion_m)-np.nanmin(p.elevacion_m)),1.0)
        distancia=((p.chi_m.to_numpy()-x)/dx)**2+((p.elevacion_m.to_numpy()-z)/dz)**2
        distancia[~np.isfinite(distancia)]=np.inf
        j=int(np.argmin(distancia));nodo=p.iloc[j]
        fila={'ID':rid,'id_kp':f'MANUAL_R{int(rid):02d}_{j}','indice':j,
              'perfil_hash':pa.profile_hash(p),'fecha':datetime.now(timezone.utc).isoformat(),
              **{k:float(nodo[k]) for k in pa.COLS}}
        tabla=tabla.loc[~tabla.ID.eq(rid)].copy()
        tabla=pd.concat([tabla,pd.DataFrame([fila])],ignore_index=True)
    destino=pa.version_path(BASE/'Archivos_Memoria/Selecciones/kp_manuales.csv')
    tabla.to_csv(destino,index=False,sep=';',decimal=',',encoding='utf-8-sig')
    config=json.loads(pa.CFG_PATH.read_text(encoding='utf-8'))
    config['kp_manual_csv']=destino.relative_to(BASE).as_posix()
    pa.save_json(pa.CFG_PATH,config)
    print('Selección nueva:',destino,'· Reiniciar 02–04 para recargarla.')
    get_ipython().run_line_magic('matplotlib','inline')